# 🏗️ Model Architecture Exploration Notebook

This notebook explores the model architectures for fair synthetic data generation.

## Contents

1. **Generator Architectures Overview**
2. **VAE-based Architectures**
3. **GAN-based Architectures**
4. **Diffusion-based Architectures**
5. **Fairness Integration Patterns**
6. **Architecture Comparison**

## Setup

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

import numpy as np
import torch
import torch.nn as nn
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print("✅ Imports successful")

## 1. Generator Architectures Overview

Three main families of generative models are used for synthetic tabular data:

| Architecture | Strengths | Weaknesses | Fair Extension |
|-------------|-----------|------------|----------------|
| **VAE** | Stable training, latent space control | Blurry outputs | Adversarial regularization |
| **GAN** | High-quality samples | Mode collapse, unstable | Fair discriminator |
| **Diffusion** | State-of-the-art quality | Slow sampling | Guided denoising |

## 2. VAE-based Architecture

The Variational Autoencoder learns a smooth latent space, making it ideal for
controlled generation with fairness constraints.

In [ ]:
class FairVAE(nn.Module):
    """VAE with adversarial fairness constraint."""

    def __init__(self, input_dim: int, latent_dim: int = 32, hidden_dim: int = 128):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_var = nn.Linear(hidden_dim, latent_dim)

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, input_dim),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_var(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var


# Instantiate and inspect
vae = FairVAE(input_dim=10, latent_dim=32, hidden_dim=128)
print(f"FairVAE parameters: {sum(p.numel() for p in vae.parameters()):,}")
print(vae)

## 3. GAN-based Architecture

A Generator-Discriminator pair with an additional **fairness critic** that
penalizes the generator for producing biased synthetic data.

In [ ]:
class Generator(nn.Module):
    """GAN generator for tabular data."""

    def __init__(self, noise_dim: int, output_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    """GAN discriminator for tabular data."""

    def __init__(self, input_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


class FairnessCritic(nn.Module):
    """Predicts sensitive attribute from generated data — used adversarially."""

    def __init__(self, input_dim: int, n_classes: int = 2, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, x):
        return self.net(x)


gen = Generator(noise_dim=32, output_dim=10)
disc = Discriminator(input_dim=10)
critic = FairnessCritic(input_dim=10)

print(f"Generator params:      {sum(p.numel() for p in gen.parameters()):,}")
print(f"Discriminator params:  {sum(p.numel() for p in disc.parameters()):,}")
print(f"FairnessCritic params: {sum(p.numel() for p in critic.parameters()):,}")

## 4. Diffusion-based Architecture (Concept)

Diffusion models iteratively denoise data from pure noise, achieving state-of-the-art
quality. For tabular data, the key idea is **guided denoising** conditioned on
fairness constraints.

### Key Concepts

- **Forward process**: Gradually add Gaussian noise to real data
- **Reverse process**: Learn to denoise step-by-step
- **Fair guidance**: At each step, steer denoising toward fair outcomes

> **Note**: Full diffusion implementation is in `src/models/diffusion/`. This
> section provides a conceptual overview only.

## 5. Fairness Integration Patterns

Three main strategies for integrating fairness into generative models:

### Pre-processing
- Modify training data before generation
- Reweighting, resampling, or relabeling

### In-processing (Recommended)
- Add fairness losses during training
- Adversarial debiasing, constrained optimization

### Post-processing
- Adjust generated data after sampling
- Threshold calibration, reject option classification

## 6. Architecture Comparison

| Metric | VAE | GAN | Diffusion |
|--------|-----|-----|-----------|
| Training stability | ✅ Stable | ⚠️ Unstable | ✅ Stable |
| Sample quality | Good | Very good | Excellent |
| Training speed | Fast | Medium | Slow |
| Latent control | ✅ Yes | ❌ No | ⚠️ Limited |
| Fairness integration | Easy (latent reg.) | Moderate (adversarial) | Complex (guidance) |
| Tabular suitability | ✅ Good | ✅ Good | ⚠️ Emerging |

In [ ]:
# Quick parameter comparison
architectures = {
    'FairVAE': sum(p.numel() for p in vae.parameters()),
    'GAN (G+D)': sum(p.numel() for p in gen.parameters()) + sum(p.numel() for p in disc.parameters()),
    'GAN + FairCritic': (
        sum(p.numel() for p in gen.parameters()) +
        sum(p.numel() for p in disc.parameters()) +
        sum(p.numel() for p in critic.parameters())
    ),
}

print("📊 Parameter Count Comparison:")
for name, count in architectures.items():
    print(f"  {name}: {count:,}")